# Britpop Merge: Spotify + MusicBrainz + AllMusic

Cilj: napraviti jedan finalni dataset `britpop_merged.csv` koji spaja
`spotify_clean.csv`, `musicbrainz_clean.csv` i `allmusic_clean.csv`.

Dva pravila:

1. **Ne izmišljamo podatke.** Podatak se upisuje samo kad je spoj siguran
   (tačan ključ ili fuzzy poklapanje iznad provernog praga).
2. **U finalnom datasetu ostaju samo redovi kod kojih su se sva tri dataseta
   uspešno spojila.** Ako neka pesma nema svoj MusicBrainz ili AllMusic
   zapis, taj red se izbacuje iz finalnog fajla (ne ostavlja se prazan).
3. **Nema dodatnih kolona.** Finalne kolone su tačno unija kolona iz tri
   ulazna fajla - nikakve pomoćne/status kolone se ne dodaju.

Redosled:
1. Učitavanje sva tri fajla.
2. Spotify + MusicBrainz - spoj preko ključa (tačan, bez nagađanja).
3. + AllMusic - fuzzy spoj (izvođač + pesma + album).
4. Izbacivanje redova kod kojih spoj nije potpun, provera i čuvanje kao
   `britpop_merged.csv`.

In [10]:
!pip install pandas rapidfuzz --break-system-packages -q

In [11]:
import re
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz

BASE_DIR = Path.cwd()

SPOTIFY_CSV = BASE_DIR / "spotify_clean.csv"
MUSICBRAINZ_CSV = BASE_DIR / "musicbrainz_clean.csv"
ALLMUSIC_CSV = BASE_DIR / "allmusic_clean.csv"
OUTPUT_CSV = BASE_DIR / "britpop_merged.csv"

print("Radni folder:", BASE_DIR)
for p in (SPOTIFY_CSV, MUSICBRAINZ_CSV, ALLMUSIC_CSV):
    print(f"  {'OK' if p.exists() else 'NEDOSTAJE'} - {p.name}")

Radni folder: C:\Users\Milica\Desktop\FON\diplomski
  OK - spotify_clean.csv
  OK - musicbrainz_clean.csv
  OK - allmusic_clean.csv


## 1) Učitavanje tri dataseta

In [12]:
spotify = pd.read_csv(SPOTIFY_CSV)
musicbrainz = pd.read_csv(MUSICBRAINZ_CSV)
allmusic = pd.read_csv(ALLMUSIC_CSV, low_memory=False)

print("spotify_clean:  ", spotify.shape)
print("musicbrainz_clean:", musicbrainz.shape)
print("allmusic_clean: ", allmusic.shape)

spotify_clean:   (2145, 26)
musicbrainz_clean: (2145, 15)
allmusic_clean:  (2313, 18)


## 2) Spajanje Spotify + MusicBrainz (spoj preko ključa)

`musicbrainz_clean.csv` već ima kolonu `spotify_track_id` koja je 1-na-1
uparena sa `track_id` iz `spotify_clean.csv` - ovo je spoj preko tačnog
ključa, bez ikakvog pogađanja.

Kod nekoliko pesama taj MusicBrainz zapis je "prazan" (`musicbrainz_mbid`
nedostaje) - to znači da MusicBrainz originalno nije uspeo da nađe tu
pesmu. Te redove za sada ostavljamo (izbacićemo ih tek na kraju, zajedno sa
onima kod kojih ni AllMusic spoj ne uspe), da bismo mogli jasno da prikažemo
koliko ih ima na svakom koraku.

In [13]:
merged = spotify.merge(
    musicbrainz.rename(columns={"spotify_track_id": "track_id"}),
    on="track_id",
    how="left",
    validate="one_to_one",
)

assert len(merged) == len(spotify), "Spoj je promenio broj redova - nesto nije u redu."

missing_mbid = merged["musicbrainz_mbid"].isna()
print(f"Redova bez musicbrainz_mbid posle key spoja: {missing_mbid.sum()} / {len(merged)}")

Redova bez musicbrainz_mbid posle key spoja: 13 / 2145


## 3) Spajanje sa AllMusic (fuzzy spoj)

AllMusic nema zajednički ključ sa Spotify-jem, pa se spaja po sličnosti
naziva: izvođač (tačno, uz podršku za kolaboracije tipa "Paul Weller, Damon
Albarn"), naziv pesme i naziv albuma. Naslovi se prvo normalizuju (mala
slova, uklanjanje "- Remastered/Live/Radio Edit" nastavaka, zagrada,
interpunkcije, "the/a/an" na početku), pa se poredi sličnost
(`rapidfuzz.token_sort_ratio`).

Konačna ocena = 85% težine na naslovu pesme + 15% na nazivu albuma (album
često nije identičan jer AllMusic zna pesmu samo sa njenog originalnog
izdanja, dok je Spotify ima i na kompilacijama/deluxe izdanjima - i dalje je
ista pesma, samo drugo izdanje).

Spoj se prihvata (pesma se smatra pronađenom) ako je naslov pesme tačno
poklopljen NAKON normalizacije, ili ako je ukupna ocena >= 70. Ovaj prag je
ručno proveren na ovom datasetu - iznad njega su gotovo uvek tačna
poklapanja, ispod njega uglavnom slučajan šum (pesme koje AllMusic
jednostavno nema, npr. razne kompilacije/B-strane).

Kolone `artist_name`, `album_title`, `song_title`, `track_number` iz
AllMusic-a se NE dodaju - one opisuju istu stvar kao već postojeće Spotify
kolone (`artist_name`, `album_name`, `track_name`, `track_number`), pa bi
njihovo dodavanje značilo ili duplikat kolone ili prepisivanje već tačnih
Spotify vrednosti. Sve OSTALE AllMusic kolone (žanr, stil, raspoloženje,
kompozitori...) se dodaju kao takve, bez izmena naziva.

In [14]:
def norm(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r"\[.*?\]", " ", s)
    s = re.sub(r"\(.*?\)", " ", s)
    s = re.sub(
        r"\s*-\s*(remaster(ed)?|live|radio edit|mono|stereo|single version|album version|edit|demo|acoustic).*$",
        " ",
        s,
    )
    s = re.sub(r"[’‘]", "'", s)
    s = re.sub(r"[^a-z0-9' ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"^(the|a|an) ", "", s)
    return s


merged["_track_key"] = merged["track_name"].apply(norm)
merged["_album_key"] = merged["album_name"].apply(norm)
allmusic["_track_key"] = allmusic["song_title"].apply(norm)
allmusic["_album_key"] = allmusic["album_title"].apply(norm)

am_by_artist = {a: g.reset_index(drop=True) for a, g in allmusic.groupby("artist_name")}

# ove 4 kolone se namerno preskacu - vec postoje (pod drugim imenom) u
# Spotify delu, videti objasnjenje iznad
AM_COLS = [
    c for c in allmusic.columns
    if c not in ("_track_key", "_album_key", "artist_name", "album_title", "song_title", "track_number")
]

MATCH_THRESHOLD = 70


def candidates_for(artist_name):
    # podrzava kolaboracije tipa "Paul Weller, Damon Albarn" - trazi kandidate
    # kod SVAKOG navedenog izvodjaca i spaja ih
    parts = [p.strip() for p in str(artist_name).split(",")]
    frames = [am_by_artist[p] for p in parts if p in am_by_artist]
    if not frames:
        return None
    return frames[0] if len(frames) == 1 else pd.concat(frames, ignore_index=True)


out_rows = []
allmusic_matched = 0

for _, row in merged.iterrows():
    candidates = candidates_for(row["artist_name"])

    best_score, best_row = -1, None
    if candidates is not None and not candidates.empty:
        track_key, album_key = row["_track_key"], row["_album_key"]
        for _, c in candidates.iterrows():
            t_score = fuzz.token_sort_ratio(track_key, c["_track_key"])
            a_score = fuzz.token_sort_ratio(album_key, c["_album_key"])
            score = 0.85 * t_score + 0.15 * a_score
            if score > best_score:
                best_score, best_row = score, c

    exact = best_row is not None and row["_track_key"] == best_row["_track_key"]
    is_match = best_row is not None and (exact or best_score >= MATCH_THRESHOLD)

    out = row.drop(["_track_key", "_album_key"]).to_dict()
    if is_match:
        for c in AM_COLS:
            out[c] = best_row[c]
        allmusic_matched += 1
    else:
        for c in AM_COLS:
            out[c] = pd.NA

    out_rows.append(out)

with_allmusic = pd.DataFrame(out_rows)
print(f"Redova sa AllMusic pogotkom: {allmusic_matched} / {len(with_allmusic)}")

Redova sa AllMusic pogotkom: 1632 / 2145


## 4) Zadržavanje samo potpuno spojenih redova, provera i čuvanje

Sada izbacujemo sve redove kod kojih BILO KOJI od dva spoja nije uspeo -
u finalnom fajlu ostaju samo pesme koje imaju i `musicbrainz_mbid` i pravi
AllMusic pogodak.

In [15]:
has_musicbrainz = with_allmusic["musicbrainz_mbid"].notna()
has_allmusic = with_allmusic[AM_COLS[0]].notna()

final = with_allmusic[has_musicbrainz & has_allmusic].reset_index(drop=True)

print(f"Spotify (bazni dataset):              {len(spotify)}")
print(f"Sa musicbrainz_mbid:                  {has_musicbrainz.sum()}")
print(f"Sa AllMusic pogotkom:                 {has_allmusic.sum()}")
print(f"Spojeno sa SVA TRI dataseta (finalno): {len(final)}")
print(f"Izbačeno (nepotpun spoj):              {len(spotify) - len(final)}")
print()
print("Broj kolona u finalnom datasetu:", final.shape[1])

Spotify (bazni dataset):              2145
Sa musicbrainz_mbid:                  2132
Sa AllMusic pogotkom:                 1632
Spojeno sa SVA TRI dataseta (finalno): 1630
Izbačeno (nepotpun spoj):              515

Broj kolona u finalnom datasetu: 54


In [16]:
final.to_csv(OUTPUT_CSV, index=False)
print(f"Sacuvano: {OUTPUT_CSV}")
print(f"Finalni oblik: {final.shape}")

Sacuvano: C:\Users\Milica\Desktop\FON\diplomski\britpop_merged.csv
Finalni oblik: (1630, 54)


# Čišćenje britpop_merged.csv

Ulaz: `britpop_merged.csv` (spojen Spotify + MusicBrainz + AllMusic).
Izlaz: `britpop.csv`.

Koraci:
1. Popunjavanje nepotpunih datuma u `release_date` iz `allmusic_release_date`
   / `musicbrainz_release_date`, pa uklanjanje te dve izvorne kolone.
2. Uklanjanje `musicbrainz_mbid`, `artist_id`, `album_id`.
3. Preimenovanje kolona (uklanjanje `musicbrainz_`/`allmusic_` prefiksa kod vecine kolona).
4. Raspoređivanje kolona u dogovoreni redosled.
5. Čuvanje kao `britpop.csv`.

In [17]:
!pip install pandas --break-system-packages -q

In [18]:
import re
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd()
INPUT_CSV = BASE_DIR / "britpop_merged.csv"
OUTPUT_CSV = BASE_DIR / "britpop.csv"

print("Radni folder:", BASE_DIR)
print(f"  {'OK' if INPUT_CSV.exists() else 'NEDOSTAJE'} - {INPUT_CSV.name}")

Radni folder: C:\Users\Milica\Desktop\FON\diplomski
  OK - britpop_merged.csv


In [19]:
df = pd.read_csv(INPUT_CSV, low_memory=False)
print("Učitano:", df.shape)

Učitano: (1630, 54)


## 2) Popunjavanje nepotpunih datuma u `release_date`

"Potpun" datum = tačno u formatu `yyyy-mm-dd` (npr. `1994-10-01`) - ništa
ne fali (ne računa se npr. samo `1994` ili `1994-10`).

Za redove kod kojih `release_date` NIJE potpun, proveravamo redom:

1. `allmusic_release_date` - ako je potpun, koristimo njega.
2. inače `musicbrainz_release_date` - ako je potpun, koristimo njega.
3. ako ni jedan nije potpun, `release_date` ostaje kakav jeste.

Na kraju ove ćelije u datasetu ostaje SAMO kolona `release_date` - obe
izvorne kolone (`allmusic_release_date`, `musicbrainz_release_date`) se
brišu.

In [20]:
DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")


def is_full_date(value):
    if pd.isna(value):
        return False
    return bool(DATE_RE.fullmatch(str(value)))


release_ok = df["release_date"].apply(is_full_date)
am_ok = df["allmusic_release_date"].apply(is_full_date)
mb_ok = df["musicbrainz_release_date"].apply(is_full_date)

needs_fill = ~release_ok
use_am = needs_fill & am_ok
use_mb = needs_fill & ~am_ok & mb_ok

df.loc[use_am, "release_date"] = df.loc[use_am, "allmusic_release_date"]
df.loc[use_mb, "release_date"] = df.loc[use_mb, "musicbrainz_release_date"]

still_incomplete = (~df["release_date"].apply(is_full_date)).sum()

print(f"Redova sa nepotpunim release_date pre popunjavanja: {needs_fill.sum()}")
print(f"  Popunjeno iz allmusic_release_date:    {use_am.sum()}")
print(f"  Popunjeno iz musicbrainz_release_date: {use_mb.sum()}")
print(f"Pesama koje NEMAJU potpun datum ni nakon popunjavanja: {still_incomplete}")

df = df.drop(columns=["allmusic_release_date", "musicbrainz_release_date"])

Redova sa nepotpunim release_date pre popunjavanja: 328
  Popunjeno iz allmusic_release_date:    267
  Popunjeno iz musicbrainz_release_date: 24
Pesama koje NEMAJU potpun datum ni nakon popunjavanja: 37


## 3) Uklanjanje, preimenovanje i rasporedjivanje kolona

- Uklanjamo `musicbrainz_mbid`, `artist_id`, `album_id`.
- Skidamo prefiks `musicbrainz_` sa svih preostalih kolona OSIM
  `musicbrainz_tags`, `musicbrainz_num_tags`, `musicbrainz_has_tags`.
- Skidamo prefiks `allmusic_` sa svih kolona OSIM `allmusic_track_pick`.
- `musicbrainz_gender` -> `artist_gender`, `musicbrainz_is_active` ->
  `artist_is_active` (posebna imena, ne samo skidanje prefiksa).

In [21]:
df = df.drop(columns=["musicbrainz_mbid", "artist_id", "album_id"])

RENAME_MAP = {
    # musicbrainz_* -> bez prefiksa (osim tags/num_tags/has_tags, ostaju isti)
    "musicbrainz_release_country": "release_country",
    "musicbrainz_label": "label",
    "musicbrainz_has_label": "has_label",
    "musicbrainz_artist_type": "artist_type",
    "musicbrainz_gender": "artist_gender",
    "musicbrainz_artist_country": "artist_country",
    "musicbrainz_artist_lifespan_begin": "artist_lifespan_begin",
    "musicbrainz_artist_lifespan_end": "artist_lifespan_end",
    "musicbrainz_is_active": "artist_is_active",
    # allmusic_* -> bez prefiksa (osim allmusic_track_pick, ostaje isti)
    "allmusic_song_genre": "song_genre",
    "allmusic_song_style": "song_style",
    "allmusic_song_mood": "song_mood",
    "allmusic_song_theme": "song_theme",
    "allmusic_composers": "composers",
    "allmusic_album_genre": "album_genre",
    "allmusic_album_style": "album_style",
    "allmusic_album_mood": "album_mood",
    "allmusic_album_theme": "album_theme",
    "allmusic_album_duration": "album_duration",
    "allmusic_recording_location": "recording_location",
    "allmusic_album_duration_sec": "album_duration_sec",
}

df = df.rename(columns=RENAME_MAP)
print("Kolona posle preimenovanja:", df.shape[1])

Kolona posle preimenovanja: 49


In [22]:
COLUMN_ORDER = [
    "track_id", "track_name", "artist_name", "album_name", "song_genre",
    "release_date", "track_number", "disc_number", "duration_ms", "duration_s",
    "release_country", "composers", "recording_location", "label", "has_label",
    "musicbrainz_tags", "musicbrainz_num_tags", "musicbrainz_has_tags",
    "song_style", "song_mood", "song_theme", "album_type", "album_genre",
    "album_style", "album_mood", "album_theme", "album_duration",
    "album_duration_sec", "artist_type", "artist_gender", "artist_country",
    "artist_lifespan_begin", "artist_lifespan_end", "artist_is_active",
    "allmusic_track_pick", "popularity", "explicit", "danceability", "energy",
    "key", "loudness", "mode", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "time_signature",
]

missing = set(COLUMN_ORDER) - set(df.columns)
extra = set(df.columns) - set(COLUMN_ORDER)
assert not missing, f"Nedostaju ocekivane kolone: {missing}"
assert not extra, f"Ima kolona koje nisu na spisku: {extra}"

df = df[COLUMN_ORDER]
print("Finalni redosled OK, broj kolona:", df.shape[1])

Finalni redosled OK, broj kolona: 49


In [23]:
# Čuvanje
df.to_csv(OUTPUT_CSV, index=False)
print(f"Sacuvano: {OUTPUT_CSV}")
print(f"Finalni oblik: {df.shape}")

Sacuvano: C:\Users\Milica\Desktop\FON\diplomski\britpop.csv
Finalni oblik: (1630, 49)
